# Recurrent Neural Networks (RNNs): From Equations to Practice

> **Goal:** Build a durable PyTorch reference for RNNs: understand recurrence and BPTT, distinguish vanilla RNNs, GRUs, and LSTMs, prepare sequences correctly, train a language model, and diagnose common issues.
>
> **Level:** Assumes familiarity with basic ML, gradients, and feed-forward networks. Equations are connected to tensor shapes and code.
>
> **Hands-on project:** A character-level language model that predicts the next character in the public Tiny Shakespeare corpus. The code uses PyTorch and Python's standard library.

This notebook is a self-contained guide. Run the cells in order; the default training budget is small so you can quickly try the full workflow. Increase `MAX_STEPS` for better samples.


## Concept map

1. **Sequence and state:** An RNN processes elements one at a time and carries a state that holds information from the past.
2. **Learning:** Parameters are shared across time; gradients flow through the network unrolled over time (BPTT).
3. **Long-range dependencies:** Products of Jacobians can cause gradients to vanish or explode. LSTMs and GRUs introduce gates.
4. **Practice:** Token encoding, shifted input/target pairs, chunk batching, gradient clipping, evaluation, and generation.
5. **Choosing a model:** A vanilla RNN is simple and fast for short dependencies; a GRU is often a good compromise; an LSTM has an explicit cell state and forget gate.

### Expected shapes

For a batch of sequences of length `T` and feature dimension `F`, the input `x` usually has shape **`(B, T, F)`** (`batch_first=True`). The hidden state has shape **`(num_layers * directions, B, H)`**. The recurrent layer output has shape **`(B, T, directions * H)`**. In this notebook, tokens are integer indices `(B,T)`, and the embedding maps them to vectors `(B,T,E)`.


## 1. What makes a network recurrent?

In a feed-forward network, the output depends on the current input. An RNN adds a **hidden state** `h_t` that summarizes the context so far:

$$
a_t = W_{xh}x_t + W_{hh}h_{t-1} + b_h,\qquad h_t = \phi(a_t)
$$
$$
y_t = W_{hy}h_t+b_y.
$$

The same matrices `W_xh`, `W_hh`, and `W_hy` are used at every time step. Sharing parameters lets the model handle sequences of different lengths. `h_0` is usually initialized to zeros or to a learned vector.

Key intuition: recurrence does not mean the model “remembers everything.” The state has finite dimensionality and is updated lossily. What remains in the state has to be learned.


In [ ]:
import math
import random
import time
import urllib.request

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)


### Unrolling through time

A sequence of length `T` can be drawn as `T` copies of the same recurrent cell. These are not separate sets of parameters: all copies share weights. Each step can emit a prediction, and the loss can sum errors across time.

- **Many-to-one:** sequence → one label (e.g. sentiment); typically classify the final state.
- **One-to-many:** one context → a sequence (e.g. generation); the state/initial token conditions subsequent steps.
- **Many-to-many:** input sequence → a prediction at every step (tagging, language modeling).

A language model learns `P(x_t | x_{<t})`. An input sequence provides context, and the output at every step is a distribution over the next token.


## 2. BPTT and long-range dependencies

**Backpropagation Through Time (BPTT)** is ordinary backpropagation through the computation graph unrolled over time. The shared recurrent matrix receives gradient contributions from many time steps.

In simplified form, a gradient from a distant step contains a product of Jacobians:

$$
\frac{\partial h_t}{\partial h_k} = \prod_{j=k+1}^{t}\frac{\partial h_j}{\partial h_{j-1}}.
$$

If the norms of successive factors are usually below 1, gradients vanish; if they are above 1, gradients can explode. Norm alone is not the whole story (direction and nonlinearities matter too), but the product intuition is essential.

**Common practical responses:** gated cells (LSTM/GRU), sensible sequence lengths, normalization/initialization, and gradient norm clipping. Clipping limits update spikes, but it does not fix the representation or create long-term memory by itself.

### Truncated BPTT

For long streams, split the sequence into chunks and detach the state between chunks (`state = state.detach()`). The state still carries context forward, but gradients do not flow back across the chunk boundary. This reduces memory and computation at the cost of limiting the learning horizon.


## 3. Recurrent cell families

### Vanilla RNN

The simplest update is $$h_t = tanh(W_x x_t + W_h h_{t-1} + b)$$. It is a useful baseline and works for short dependencies, but learning long context is difficult.

### LSTM

An LSTM maintains a **hidden state** `h_t` and a **cell state** `c_t`. Gates control forgetting, writing, and reading:

$$
i_t=\sigma(W_i[x_t,h_{t-1}]+b_i),\quad f_t=\sigma(W_f[x_t,h_{t-1}]+b_f),\quad o_t=\sigma(W_o[x_t,h_{t-1}]+b_o)
$$
$$
\tilde c_t=\tanh(W_c[x_t,h_{t-1}]+b_c),\quad c_t=f_t\odot c_{t-1}+i_t\odot\tilde c_t,\quad h_t=o_t\odot\tanh(c_t).
$$

The additive path in `c_t` helps gradients flow. The gates are learned; they are not manually assigned semantic memory.

### GRU

A GRU combines memory into a single state `h_t` and uses update and reset gates. It typically has fewer parameters than an LSTM and often performs similarly well. There is no universal winner—compare them on your task with a similar tuning budget.

### Bidirectional and multilayer networks

- `num_layers > 1` stacks recurrent cells; dropout in `nn.RNN/GRU/LSTM` is applied between layers (and is ignored for a single layer).
- `bidirectional=True` processes a sequence in both directions. This is useful for classification/tagging when the whole sequence is available. **It is not suitable for causal generation**, because the backward direction sees the future.


## 4. Environment setup and public dataset

The corpus is loaded from the public [karpathy/char-rnn](https://github.com/karpathy/char-rnn) repository directly into memory. The first run requires network access. If you are offline, save the corpus locally and replace the download cell with a read from that file.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
print("Loading Tiny Shakespeare into memory…")
with urllib.request.urlopen(DATA_URL) as response:
    text = response.read().decode("utf-8")
print(f"Characters: {len(text):,}")
print(text[:500])


## 5. Character tokenization and data split

Character-level tokenization is simple and makes autoregression visible without an extra library. The vocabulary contains only characters present in the corpus; production natural-language systems usually use a subword tokenizer.

The split is **sequential**: the first 90% is training data and the remaining 10% is validation data. This reflects predicting future text and prevents neighboring chunks from being randomly mixed across splits. There is no separate test set here—the goal is to study the model, not produce a publishable benchmark.


In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encoded = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

split = int(0.9 * len(encoded))
train_ids = encoded[:split]
val_ids = encoded[split:]
print("Vocab size:", vocab_size)
print("Train / val tokens:", len(train_ids), "/", len(val_ids))
print("Sample token IDs:", train_ids[:20].tolist())


### A prediction task without leakage

For a chunk containing tokens `a b c d`, the input is `a b c` and the targets are `b c d`. The model predicts the next character at every position. Cross-entropy compares logits `(B,T,V)` with labels `(B,T)`; the first two dimensions are flattened before computing the loss.

The generator below samples chunk start positions from within a single split. Shifting by one preserves context within each chunk.


In [ ]:
BATCH_SIZE = 64
BLOCK_SIZE = 128


def get_batch(split_name, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE):
    source = train_ids if split_name == "train" else val_ids
    starts = torch.randint(len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in starts])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in starts])
    return x.to(device), y.to(device)

xb, yb = get_batch("train", batch_size=2, block_size=12)
print("x:", xb.shape, "y:", yb.shape)
print("input: ", "".join(itos[i] for i in xb[0].tolist()))
print("target:", "".join(itos[i] for i in yb[0].tolist()))


## 6. The model in PyTorch

`nn.RNN`, `nn.GRU`, and `nn.LSTM` share a common PyTorch interface. The built-in implementations are optimized; it is still useful to know the equations to understand what `hidden_size`, state, and shapes mean.

The architecture below is **embedding → recurrent cell → vocabulary projection**. Logits are unnormalized scores; `CrossEntropyLoss` applies a numerically stable log-softmax internally. Do not add `softmax` before this loss.


In [ ]:
class CharRNN(nn.Module):
    def __init__(self, cell="LSTM", embedding_dim=64, hidden_size=128, num_layers=2, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        cell_cls = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}[cell]
        self.rnn = cell_cls(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_size, vocab_size)
        self.cell_name = cell

    def forward(self, idx, state=None):
        emb = self.embedding(idx)              # (B, T) -> (B, T, E)
        out, state = self.rnn(emb, state)      # (B, T, H)
        logits = self.head(out)                # (B, T, V)
        return logits, state

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Autoregressively appends tokens; idx has shape (B, context)."""
        self.eval()
        for _ in range(max_new_tokens):
            context = idx[:, -BLOCK_SIZE:]
            logits, _ = self(context)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k is not None:
                k = min(top_k, logits.size(-1))
                threshold = torch.topk(logits, k).values[:, [-1]]
                logits = logits.masked_fill(logits < threshold, float("-inf"))
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

model = CharRNN().to(device)
logits, state = model(xb)
print("logits:", logits.shape)
if isinstance(state, tuple):
    print("h_n, c_n:", state[0].shape, state[1].shape)
else:
    print("h_n:", state.shape)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")


### State interface and shape pitfalls

- `nn.LSTM` returns `(output, (h_n, c_n))`; `nn.RNN/GRU` return `(output, h_n)`.
- With `batch_first=True`, the `input` and `output` shapes change, **but the hidden state shape does not**.
- For bidirectional networks, the direction dimension comes first in the state, while the last dimension of `output` is `directions * hidden_size`.
- Each batch here contains independent windows, so the state defaults to zero. In a streaming setup, you can pass it between chunks, but you must preserve the order of examples and detach the graph with `.detach()`.
- `pack_padded_sequence` / `pad_packed_sequence` are useful for batches of sequences with different lengths. Alternatively, mask padding positions in the loss (`ignore_index`). All windows here have the same length.


## 7. Training, evaluation, and diagnostics

Language-model perplexity is `exp(mean cross-entropy)` and can be interpreted intuitively as the effective number of possible tokens. Compare it only when the tokenizer and data split are the same.

The training loop uses AdamW and global gradient norm clipping. Loss is estimated in `eval()` and `no_grad()` mode on several random batches. The small step budget demonstrates the workflow but is not a full tuning run.


In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_batches=20):
    model.eval()
    result = {}
    for split_name in ("train", "val"):
        losses = []
        for _ in range(eval_batches):
            x, y = get_batch(split_name)
            logits, _ = model(x)
            loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
            losses.append(loss.item())
        result[split_name] = sum(losses) / len(losses)
    model.train()
    return result


def train_model(cell, max_steps=120, eval_interval=30, lr=3e-3):
    net = CharRNN(cell=cell).to(device)
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr)
    history = {"step": [], "train": [], "val": []}
    start_time = time.time()
    net.train()
    for step in range(max_steps + 1):
        if step % eval_interval == 0 or step == max_steps:
            losses = estimate_loss(net)
            history["step"].append(step)
            history["train"].append(losses["train"])
            history["val"].append(losses["val"])
            print(f"{cell:4s} | step {step:4d} | train {losses['train']:.3f} | val {losses['val']:.3f}")
        if step == max_steps:
            break
        x, y = get_batch("train")
        logits, _ = net(x)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()
    print(f"elapsed: {time.time() - start_time:.1f}s")
    return net, history


### Comparing RNN, GRU, and LSTM

We use the same width, embedding size, data split, and number of optimization steps. Parameter counts will differ because the cells have different numbers of gates. This is a quick educational comparison, not a conclusive experiment: for a reliable result, repeat with several seeds, tune hyperparameters, and average the results.

Training all three variants on CPU may take a while. Reduce `MAX_STEPS` to 40 for a quick smoke run, or increase it to several hundred for more interesting samples.


In [ ]:
MAX_STEPS = 120
models = {}
histories = {}
for cell in ("RNN", "GRU", "LSTM"):
    models[cell], histories[cell] = train_model(cell, max_steps=MAX_STEPS)


In [ ]:
plt.figure(figsize=(9, 5))
for cell, hist in histories.items():
    plt.plot(hist["step"], hist["val"], marker="o", label=cell)
plt.xlabel("Optimization step")
plt.ylabel("Validation cross-entropy (nats/character)")
plt.title("Comparison of recurrent cells")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

for cell, net in models.items():
    print(f"{cell}: {sum(p.numel() for p in net.parameters()):,} parameters")


### How to read the curves

- Decreasing training and validation losses suggest that the model is using local patterns in the text.
- A widening train–validation gap may indicate overfitting; reduce capacity, add dropout, train for fewer steps, or gather more data.
- A small difference between cells with a small training budget does not prove they are equivalent. Optimization and random seeds introduce noise.
- For a uniform random predictor, the expected loss is `log(vocab_size)`. A learning model should do better.

Also track gradient norms, time per step, GPU memory, and sample quality. Loss alone does not tell you everything about the usefulness of generated text.


## 8. Generation and temperature

Generation starts with a seed character. At each step, the model returns logits for the next character; softmax gives a distribution, and `multinomial` samples a token. Temperature `τ` transforms logits to `z/τ`: a low temperature boosts the most likely options, while a high temperature flattens the distribution. `top_k` keeps only the `k` most likely tokens.

Sampling changes diversity, not the model's knowledge. Greedy decoding (`argmax`) is deterministic and often gets stuck in loops.


In [ ]:
def sample_text(cell="LSTM", prompt="ROMEO:", length=800, temperature=0.8, top_k=20):
    net = models[cell]
    prompt_ids = [stoi[ch] for ch in prompt if ch in stoi]
    if not prompt_ids:
        prompt_ids = [0]
    seed = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    generated = net.generate(seed, length, temperature=temperature, top_k=top_k)[0].tolist()
    return "".join(itos[i] for i in generated)

print(sample_text("LSTM", prompt="ROMEO:", length=800, temperature=0.8, top_k=20))


In [ ]:
for temp in (0.4, 0.8, 1.2):
    print(f"\n--- temperature {temp} ---")
    print(sample_text("LSTM", prompt="KING:", length=300, temperature=temp, top_k=30))


## 9. Practical considerations and variants

### Variable lengths and padding

Text batches with different sequence lengths are often padded. A vanilla RNN also updates its state on padding, which can distort the final state. Use `pack_padded_sequence` for classification or mask padding positions in the loss. Pay attention to `lengths` (CPU for some APIs) and sorting when `enforce_sorted=True`.

### Sequence classification

For a label assigned to an entire text, use the final state (accounting for directions and layers) or pool over outputs. With padding, use the state for the true sequence length. A bidirectional GRU/LSTM often fits offline tasks where the full text is available.

### Continuous-value prediction

For time-series forecasting, the input can be `(B,T,F)` with real-valued features, and a linear head predicts the next value. The temporal split must preserve chronology. Fit scaling only on training data, avoid future leakage, compare with a naive baseline, and evaluate at the horizon that matters for the application.

### Stateful vs. stateless

The windows here are independent (stateless). In streaming training, you can carry the state to the next chunk, but each batch slot must continue the same stream. After reordering or shuffling examples, reset or correctly reassign the state. Carrying state without tracking it is a common source of silent bugs.

### Performance and limitations

Recurrence makes step `t` depend on step `t-1`, so it is harder to parallelize over time than convolutions or Transformers. Embeddings let us compute input representations in parallel, but hidden-state steps remain sequential. RNNs are still useful for streaming, low-memory settings, short sequences, and existing systems.


## 10. Good practices and common mistakes

| Problem | Symptom | Check / fix |
|---|---|---|
| Incorrect target shift | model copies the input or loss is meaningless | `y = x` shifted by exactly one token |
| Wrong tensor axis | odd shapes or batch treated as time | set `batch_first=True` or reorder dimensions |
| Softmax before CrossEntropyLoss | worse numerical stability and incorrect scale | pass raw logits |
| Exploding gradients | NaNs, sudden loss spikes | check data/LR and clip gradient norm |
| State carried between unrelated batches | unstable validation, leakage | reset state for independent windows |
| Bidirectional model used for generation | future context is unavailable at inference | use a unidirectional causal model |
| Evaluating on train instead of val | overly optimistic results | use a separate chronological validation split |
| Accidental chronological mixing | leakage in time-series tasks | split by time first, then build windows |
| Missing `eval()` / `no_grad()` | dropout remains active, unnecessary graph | switch modes for inference and evaluation |

### Pre-training checklist

- Is `x[:, t]` paired with target `x[:, t+1]`?
- Is the data split created before building windows?
- Does the final logits dimension match the vocabulary/classes?
- Does the loss receive the expected axes and data types?
- Is the baseline sensible, and is validation performance reported?
- Are the seed, tokenizer, split, and configuration recorded?


## Quick-reference summary

- An RNN updates its state: `h_t = f(x_t, h_{t-1})`; weights are shared across time.
- BPTT computes gradients through the unrolled graph; long products of Jacobians cause vanishing/exploding gradients.
- An LSTM has a cell state and gates; a GRU has a simpler gating mechanism. The choice depends on the data and budget.
- For a causal language model, input and target differ by one token; loss is computed at each position.
- `batch_first=True` means `(B,T,F)` for input and output, but the state remains `(layers*directions,B,H)`.
- Clipping limits exploding gradients; validation, a correct split, and careful state handling help prevent misleading conclusions.
- RNNs are sequential over time; bidirectional models are unsuitable for autoregressive generation.
